## LESHI example workflow

Needed installations:

In [ ]:
pip install numpy scipy pandas emcee p_tqdm astropy spectral_cube matplotlib photutils wget astrocut

For the code to work, this notebook has to be in the same directory as the "LESHI_scripts" folder.

In [1]:
from LESHI_scripts import LESHI
import pandas as pd

we can print the LESHI function helper description:

In [2]:
help(LESHI.source_finder)

Help on function source_finder in module LESHI_scripts.LESHI:

source_finder(data_cube, path_to_results, SNR_integ=3.5, SNR_channel=2.5, channel_min_len=3, SNR_spec=3.5, rsqr_min=0.35, int_image_len=10, int_image_load_no=10, channel_start=0, channel_end=None, beam=None, bg_box_size=100, max_dist_pix=10, max_dist_channel=10, test_hist=False)
    Start the LESHI Source Finder.

    Parameters
    ----------
    data_file : str
        Path to the datacube to perform source finding on.

    path_to_results : str
        Path to the directory where the results will be saved.

    SNR_integ : int or float
        Threshold signal-to-noise ratio of each source on the integrated image (default = 3.5)

    SNR_channel : int or float
        Threshold signal-to-noise ratio of each source on the channel image (default = 2.5)

    channel_min_len : int
        Minimum number of channels that the signal persists for (default = 3)

    SNR_spec : int or float
        Threshold signal-to-noise ratio

# Source finding

In [ ]:
# let's call the sourcefinder
data_cube= 
data_table_found_sources = LESHI.source_finder(
                            data_cube,
                            path_to_results='./',
                            SNR_integ=3.5, 
                            SNR_channel=2.5, 
                            channel_min_len=3,
                            SNR_spec=3.5, 
                            rsqr_min=0.35,
                            int_image_len=4, 
                            int_image_load_no=30,
                            channel_start=0, 
                            channel_end=None, 
                            beam=None, 
                            bg_box_size=100, 
                            max_dist_pix=10, 
                            max_dist_channel=10, 
                            test_hist=True )

# Example settings:

# good optimum between reliability and completeness
# SNR_integ=3.5., 
# SNR_channel=2.5, 
# channel_min_len=3,
# SNR_spec=3.5, 
# rsqr_min=0.35

# for high completeness:
# SNR_integ=3., 
# SNR_channel=2., 
# channel_min_len=3,
# SNR_spec=3., 
# rsqr_min=0.3

# for high reliability:
# SNR_integ=4., 
# SNR_channel=2.5, 
# channel_min_len=3,
# SNR_spec=3.5, 
# rsqr_min=0.4


LESHI struggles to properly associate very extended sources, unless told to associate sources with the max_dist_pix and max_dist_channel inputs, however if these values are too big, some neighbouring galaxies might be associated when they should not be. The package includes a script that finds the full extent of the sources as described in Section 4.4 of Maksymowicz-Maciata et al. 2026 and associates sources for which emission distribution overlaps in 3D. It is longish (~15 seconds per source), but is pararellised and gives good results.

In [4]:
data_table_found_sources = pd.read_csv('./source_finding_results_1/found_sources_associated.csv')
data_table_source_extent = LESHI.source_extent(data_table = data_table_found_sources, 
                                               data_cube=data_cube, 
                                               min_window_width_pix=200, 
                                               beam_diam_arc=14)

100%|██████████| 72/72 [03:14<00:00,  2.70s/it] 


# Detection plots

Now that we have the extent of the sources, we can plot the detections to check and judge if they are real or not. The package has a script to download optical images to check if there is an optical counterpart and how the detected HI emission is ditributed against the optical. For now the possible surveys are the LegacySurvey and HSC (Hyper Suprime Cam), the patch of sky covered by these surveys can be check using https://www.legacysurvey.org//viewer 

In [ ]:
# for each detection we can download optical images (curently the code works for 'HSC' and 'LegacySurvey')
# this requires the wget package to be installed (https://formulae.brew.sh/formula/wget)
# the script formulates a link that downloads the needed cutout and uses wget command to do it

data_table = data_table_source_extent

ra_array = data_table['RA_deg'].values
dec_array = data_table['Dec_deg'].values
ID_array = data_table['ID'].values # the images will be saved under the IDs from the table

# we would like the optical image to contain the whole size of HI emission
width_arc_array = data_table['contour_diameter_arc'].values*2
width_arc_array[width_arc_array<160] = 160

filters = ['G','R','I'] # images for which filters we would like to download

LESHI.get_opt_cutout(ID=ID_array,ra=ra_array,dec=dec_array,width_arc=width_arc_array,
                     filters=filters,survey='HSC',path_to_images='./optical_images/')



In [1]:
from LESHI_scripts import LESHI
import pandas as pd

# having optical images we can create detection plots showing the detection, spectrum and optical image 

data_table = pd.read_csv('found_sources_extent.csv')
LESHI.emission_plot(data_table = data_table, path_to_data_cube = data_cube, 
                     path_to_optical_images = './optical_images/', path_to_results='./',
                   image_arc_width = 100, spectrum_length = 200, beam = None, filetype='png')


100%|██████████| 51/51 [00:16<00:00,  3.03it/s]


To quickly look through and flag the detections, the package includes a script that opens an interactive window displaying the figure for each detection, which then can be flagged using keyboard input and all the flags can be saved into a table. The package needed is open-cv which can be installed with a script below. Unfortunately, since the script opens an interactive widow, it does not straightforwardly work on IDIA. 

In [ ]:
pip install opencv-python

In [ ]:
# we can go through all the detections using the eye_checker window and flag each source quickly
LESHI.eye_check(data_table = data_table, path_to_figures = './detection_plots_1/', path_to_results = './')



In [ ]:
# the above tool requires an interactive window to be opened which does not work on IDIA, but we can zip
# the plots and download them to look through them locally

import shutil
shutil.make_archive('emission_plots', 'zip', path_to_results+'emission_plots/')